# 第 6 章(加餐):加载 GPT-2 官方权重

**意义**:自己训的小模型受限于语料和算力,生成质量有限。但我们写的
`GPTModel` 架构和 OpenAI GPT-2 完全一致(只要能把权重对应上),
就能直接吃进官方预训练权重,瞬间拥有 GPT-2 的生成能力。

这一章做的就是:**参数名映射** —— 把 HuggingFace 的 GPT-2 权重搬到我们的模型里。

**章节内容**
1. 安装 transformers + 下载 GPT-2 small 权重
2. 对比两边的参数名和 shape
3. 写权重映射函数 `load_weights_into_gpt`
4. 加载后对同一 prompt 生成,验证行为一致

**需要下载**:约 500MB(GPT-2 small 权重),首次运行会花几分钟。

In [ ]:
# 如果没装 transformers,取消注释运行:
# !pip install transformers

import sys, pathlib, os
sys.path.insert(0, str(pathlib.Path('..').resolve()))
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')

import torch
import numpy as np
from transformers import GPT2LMHeadModel

from llm.tokenizer import Tokenizer
from llm.model     import GPTModel, GPT_CONFIG_124M
from llm.generate  import generate

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

---
## 1. 下载 GPT-2 small 权重

In [ ]:
print('下载中(首次约 500MB) ...')
gpt_hf = GPT2LMHeadModel.from_pretrained('gpt2')
gpt_hf.eval()
print('加载完成')

# 看看它的参数名和 shape
for name, p in list(gpt_hf.named_parameters())[:15]:
    print(f'{name:50s} {tuple(p.shape)}')

---
## 2. 对比我们自己的 GPTModel

**注意**:GPT-2 官方使用 **1D 卷积(Conv1D)代替 Linear**,权重是**转置关系**,
映射时记得 `.T`。此外 GPT-2 的 bias 在 QKV 投影里是启用的,所以我们需要
用 `qkv_bias=True` 的配置。

In [ ]:
# GPT-2 small 在我们的框架下对应配置
CFG = GPT_CONFIG_124M.copy()
CFG['qkv_bias'] = True   # GPT-2 的 QKV 投影有 bias
CFG['drop_rate'] = 0.0   # 推理时不用 dropout

model = GPTModel(CFG).to(device)
model.eval()

# 打印我们模型的参数名
print('=== 我们的 GPTModel 参数(前 15 个) ===')
for name, p in list(model.named_parameters())[:15]:
    print(f'{name:50s} {tuple(p.shape)}')

---
## 3. 权重映射

HuggingFace 参数 → 我们的参数 对应表(以第 0 层为例):

| HuggingFace                              | 我们的                                | 备注              |
|------------------------------------------|---------------------------------------|-------------------|
| `wte.weight`                             | `tok_emb.weight`                       |                   |
| `wpe.weight`                             | `pos_emb.weight`                       |                   |
| `h.0.ln_1.weight`                        | `blocks.0.ln1.scale`                   |                   |
| `h.0.ln_1.bias`                          | `blocks.0.ln1.shift`                   |                   |
| `h.0.attn.c_attn.weight` (d_in, 3*d_out) | `blocks.0.attn.W_Q/W_K/W_V.weight`     | 切三份 + 转置     |
| `h.0.attn.c_attn.bias`                   | `blocks.0.attn.W_Q/W_K/W_V.bias`       | 切三份            |
| `h.0.attn.c_proj.weight`                 | `blocks.0.attn.W_O.weight`             | 转置              |
| `h.0.attn.c_proj.bias`                   | `blocks.0.attn.W_O.bias`               |                   |
| `h.0.ln_2.weight` / `bias`               | `blocks.0.ln2.scale` / `shift`         |                   |
| `h.0.mlp.c_fc.weight` / `bias`           | `blocks.0.ffn.net[0].weight` / `bias`  | 转置              |
| `h.0.mlp.c_proj.weight` / `bias`         | `blocks.0.ffn.net[2].weight` / `bias`  | 转置              |
| `ln_f.weight` / `bias`                   | `final_ln.scale` / `shift`             |                   |
| (GPT-2 权重共享)                         | `out_head.weight`                      | 复用 `wte.weight` |

In [ ]:
def _assign(target, value):
    if target.shape != value.shape:
        raise ValueError(f'shape mismatch: {target.shape} vs {value.shape}')
    target.data = value.clone().detach()


def load_gpt2_weights(model, gpt_hf):
    """把 HuggingFace GPT2LMHeadModel 的权重搬到我们的 GPTModel。"""
    sd = gpt_hf.state_dict()

    _assign(model.tok_emb.weight, sd['transformer.wte.weight'])
    _assign(model.pos_emb.weight, sd['transformer.wpe.weight'])

    for i in range(model.cfg['n_layers']):
        b = model.blocks[i]
        p = f'transformer.h.{i}'

        # LN1
        _assign(b.ln1.scale, sd[f'{p}.ln_1.weight'])
        _assign(b.ln1.shift, sd[f'{p}.ln_1.bias'])

        # QKV 合并权重 (d_in, 3*d_out) 切成三份并转置
        qkv_w = sd[f'{p}.attn.c_attn.weight']     # (d_in, 3*d_out)
        qkv_b = sd[f'{p}.attn.c_attn.bias']       # (3*d_out,)
        d = model.cfg['emb_dim']
        q_w, k_w, v_w = qkv_w.split(d, dim=1)
        q_b, k_b, v_b = qkv_b.split(d, dim=0)
        # Conv1D 是行列颠倒的 Linear,nn.Linear 用的是 (out, in),所以 .T
        _assign(b.attn.W_Q.weight, q_w.T); _assign(b.attn.W_Q.bias, q_b)
        _assign(b.attn.W_K.weight, k_w.T); _assign(b.attn.W_K.bias, k_b)
        _assign(b.attn.W_V.weight, v_w.T); _assign(b.attn.W_V.bias, v_b)

        # 输出投影
        _assign(b.attn.W_O.weight, sd[f'{p}.attn.c_proj.weight'].T)
        _assign(b.attn.W_O.bias,   sd[f'{p}.attn.c_proj.bias'])

        # LN2
        _assign(b.ln2.scale, sd[f'{p}.ln_2.weight'])
        _assign(b.ln2.shift, sd[f'{p}.ln_2.bias'])

        # FFN
        _assign(b.ffn.net[0].weight, sd[f'{p}.mlp.c_fc.weight'].T)
        _assign(b.ffn.net[0].bias,   sd[f'{p}.mlp.c_fc.bias'])
        _assign(b.ffn.net[2].weight, sd[f'{p}.mlp.c_proj.weight'].T)
        _assign(b.ffn.net[2].bias,   sd[f'{p}.mlp.c_proj.bias'])

    # final LayerNorm
    _assign(model.final_ln.scale, sd['transformer.ln_f.weight'])
    _assign(model.final_ln.shift, sd['transformer.ln_f.bias'])

    # 权重共享:out_head 复用 wte
    _assign(model.out_head.weight, sd['transformer.wte.weight'])

    return model


load_gpt2_weights(model, gpt_hf)
print('权重加载完成')

---
## 4. 生成对比

现在用我们的模型生成,应该能产出流畅的英文。

In [ ]:
tokenizer = Tokenizer()
prompt = 'Once upon a time, there was a little dragon who'
ids = torch.tensor([tokenizer.encode(prompt)], device=device)

# 贪心生成,结果应该和 HuggingFace 官方 greedy 输出一致
out_ours = generate(model, ids, max_new_tokens=60,
                    context_length=CFG['context_length'],
                    temperature=0.01)
print('=== 我们的 GPTModel(加载权重后) ===')
print(tokenizer.decode(out_ours[0].tolist()))

# 用 HuggingFace 官方模型对比
gpt_hf.to(device)
with torch.no_grad():
    out_hf = gpt_hf.generate(ids, max_new_tokens=60, do_sample=False, pad_token_id=50256)
print('\n=== HuggingFace GPT-2 (greedy) ===')
print(tokenizer.decode(out_hf[0].tolist()))

In [ ]:
# 更严格的验证:前向一次,两个模型的 logits 应几乎完全一致
model.eval()
gpt_hf.eval()
with torch.no_grad():
    logits_ours = model(ids)                        # (1, T, V)
    logits_hf   = gpt_hf(ids).logits                # (1, T, V)

diff = (logits_ours - logits_hf).abs().max().item()
print(f'logits 最大绝对差: {diff:.6f}')
print(f'(应 < 1e-3,若如此则两模型数值等价)')

---
## 章末思考题

1. 为什么 HuggingFace 的 `c_attn.weight` 形状是 `(d_in, 3*d_out)`,而我们的 `nn.Linear` 是 `(d_out, d_in)`?这种「合并 QKV」的做法有什么好处?
2. 如果想加载 GPT-2 medium(355M)/ large(774M),配置里要改什么?(提示:layers、heads、emb_dim)
3. 加载完权重,我们是否还能对这个模型做「继续训练」(持续预训练 / 微调)?怎么做?

---

## 恭喜 🎉

到这里,你已经完成了整个「从零构建大语言模型」的学习:

- ✅ Tokenization —— 文本 ↔ 整数
- ✅ DataLoader + Embedding —— 数据管道
- ✅ Self-Attention / Causal / Multi-head —— Transformer 的心脏
- ✅ LayerNorm / GELU / FFN / TransformerBlock / GPTModel —— 完整架构
- ✅ 预训练 + 采样 —— 让模型真正会生成
- ✅ 加载 GPT-2 权重 —— 验证架构与官方等价

你现在理解了市面上所有 GPT-风格模型(GPT-2/3/4、LLaMA、Mistral 等)的核心设计。
接下来可以探索的方向:

- **SFT / 指令微调**:让预训练模型会回答问题(alpaca、dolly 等指令数据集)
- **RLHF / DPO**:让回答更符合人类偏好
- **架构升级**:RoPE 位置编码、RMSNorm、SwiGLU、分组注意力(GQA)—— LLaMA 等现代模型用的
- **工程优化**:FlashAttention、KV cache、量化(int8/int4)、模型并行